# How to navigate the iNaturalist competition datasets

Updated version of the 2024 tutorial `How to Navigate iNaturalist (1).ipynb` (archived verbatim,
with its outputs, under `archive/2024-original/notebooks/`). Written by Ryan Junejo during a
summer 2024 research internship at the Vector Institute supervised by Graham Taylor.

The code below uses the shared helpers in `src/biodiv/inat.py`, so the same category mapping and
dataset class that the training code uses are the ones you explore here.

## Where the data lives

On the Vector cluster the competition datasets were mounted read-only under
`/datasets/inat_comp/<year>/`. Elsewhere, download them from the
[visipedia/inat_comp](https://github.com/visipedia/inat_comp) release pages (the 2019 and 2021
READMEs list every archive with its md5) and point `BIODIV_DATA_ROOT` at the parent directory
so that `inat_comp/2019/...` and `inat_comp/2021/...` resolve.

| year | species | train | val | layout |
|---|---|---|---|---|
| 2019 | 1,010 | 265,213 | 3,030 | `train_val2019/<supercategory>/<category id>/<hash>.jpg`; train and val share folders, the split lives in `train2019.json` / `val2019.json` |
| 2021 | 10,000 | 2,686,843 (`train_mini`: 500,000) | 100,000 | `train/<id>_<Kingdom>_..._<species>/<uuid>.jpg`; folder names carry the taxonomy |

(2019 total of 268,243 train+val images and 2021 counts from the upstream READMEs.)

Both years use COCO-style JSON with `images`, `annotations` (one per image, `category_id`) and
`categories`. In 2019 the species names inside the split files are obfuscated; the real names are
in a separate `categories.json`.


> **Smoke test vs. real run.** When the dataset is not present under `BIODIV_DATA_ROOT`
> (or `BIODIV_SMOKE_TEST=1` is set) this notebook runs on a tiny synthetic fixture with a
> weight-free stand-in encoder. That exercises the code path only; the numbers it prints are
> **not** results. A real run needs the dataset and downloads the pinned BioCLIP checkpoint
> (~600 MB per model) from the Hugging Face Hub.

In [ ]:
import json, os, sys
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path[:0] = [str(REPO / "src"), str(REPO)]
os.environ.setdefault("BIODIV_DATA_ROOT", str(REPO / "data"))      # where inat_comp/<year>/ lives
os.environ.setdefault("BIODIV_OUTPUT_DIR", str(REPO / "outputs"))

from biodiv.paths import get_paths
paths = get_paths()  # BIODIV_DATA_ROOT / BIODIV_CACHE_DIR / BIODIV_OUTPUT_DIR
print("data_root =", paths.data_root.resolve())

In [ ]:
YEAR = os.environ.get("BIODIV_INAT_YEAR", "2019")
if YEAR == "2019":
    TRAIN_JSON, VAL_JSON = paths.data_root / "inat_comp/2019/train2019.json", paths.data_root / "inat_comp/2019/val2019.json"
    CATEGORIES_JSON, IMAGE_ROOT = paths.data_root / "inat_comp/2019/categories.json", paths.data_root / "inat_comp/2019"
else:
    TRAIN_JSON, VAL_JSON = paths.data_root / "inat_comp/2021/train_mini.json", paths.data_root / "inat_comp/2021/val.json"
    CATEGORIES_JSON, IMAGE_ROOT = None, paths.data_root / "inat_comp/2021"

SMOKE = os.environ.get("BIODIV_SMOKE_TEST", "auto")
if SMOKE == "auto":
    SMOKE = "0" if TRAIN_JSON.exists() else "1"
SMOKE = SMOKE == "1"
if SMOKE:
    import tempfile
    from biodiv.fixtures import make_inat_fixture
    fx = make_inat_fixture(Path(tempfile.mkdtemp()) / "inat")
    TRAIN_JSON, VAL_JSON, CATEGORIES_JSON, IMAGE_ROOT = fx["train"], fx["val"], fx["categories"], fx["image_root"]
print("mode:", "SMOKE TEST (synthetic fixture)" if SMOKE else f"REAL DATA ({YEAR})")

## Looking at the folder layout

Walking a whole year prints tens of thousands of lines, so this helper stops at a depth of two
and only summarises counts.

In [ ]:
def summarise_dir(root, max_depth=2, max_entries=8):
    root = Path(root)
    for path in sorted(p for p in root.rglob("*") if p.is_dir()):
        depth = len(path.relative_to(root).parts)
        if depth > max_depth:
            continue
        files = sum(1 for p in path.iterdir() if p.is_file())
        dirs = sum(1 for p in path.iterdir() if p.is_dir())
        print(f"{'  ' * depth}{path.name}/  {dirs} dirs, {files} files")
        if depth == max_depth:
            pass

summarise_dir(IMAGE_ROOT if SMOKE else IMAGE_ROOT / ("train_val2019" if YEAR == "2019" else "val"), max_depth=1)

## Loading the annotations and building an explicit category map

`load_split` returns the raw JSON plus a `CategoryMap` that converts dataset category ids to
contiguous class indices and back. The training and validation maps are compared with
`assert_same`, which is how the training code guards against label mismatches.

In [ ]:
from collections import Counter
from biodiv.inat import load_split, build_records, INatDataset

train_data, train_map = load_split(TRAIN_JSON, CATEGORIES_JSON)
val_data, val_map = load_split(VAL_JSON, CATEGORIES_JSON)
train_map.assert_same(val_map)
print(f"{train_map.num_classes} classes; ids contiguous: {train_map.is_contiguous()}")
print("first categories:", list(zip(train_map.category_ids[:5], train_map.names[:5])))

train_records = build_records(train_data, train_map)
val_records = build_records(val_data, val_map)
print(f"{len(train_records)} training records, {len(val_records)} validation records")
print("example record (file_name, class index):", train_records[0])

## Class imbalance

iNat 2019 is long-tailed (the upstream README shows the distribution); iNat 2021 `train_mini` is
exactly balanced at 50 images per species and `val` at 10. Counting labels in the records is the
quickest way to see this for the split you loaded.

In [ ]:
counts = Counter(label for _, label in train_records)
most, least = counts.most_common(3), counts.most_common()[-3:]
print("largest classes :", [(train_map.names[i], n) for i, n in most])
print("smallest classes:", [(train_map.names[i], n) for i, n in least])
print(f"min {min(counts.values())}, max {max(counts.values())}, mean {sum(counts.values()) / len(counts):.1f} images per class")

## Viewing an image, raw and through the training / evaluation transforms

`build_train_transform` is the augmentation pipeline used for fine-tuning (resize, random flip,
RandAugment); `build_eval_transform` is deterministic. The 2024 scripts evaluated with the
training pipeline by mistake; keep them separate.

In [ ]:
import random
import matplotlib.pyplot as plt
from biodiv.transforms import build_eval_transform, build_train_transform, IMAGENET_MEAN, IMAGENET_STD
import torch

random.seed(15)
dataset_raw = INatDataset(val_records, IMAGE_ROOT, transform=None)
idx = random.randrange(len(dataset_raw))
image, label = dataset_raw[idx]
print(f"record {idx}: class index {label} -> category id {val_map.to_category_id(label)} ({val_map.names[label]}), size {image.size}")

def to_display(t):  # undo normalisation for plotting
    mean, std = torch.tensor(IMAGENET_MEAN)[:, None, None], torch.tensor(IMAGENET_STD)[:, None, None]
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0)

size = 64 if SMOKE else 224
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(image); axes[0].set_title("original")
axes[1].imshow(to_display(build_eval_transform(size)(image))); axes[1].set_title("eval transform")
axes[2].imshow(to_display(build_train_transform(size)(image))); axes[2].set_title("train transform")
for ax in axes:
    ax.axis("off")
plt.show()

## Terms of use

The iNaturalist competition data may be used for non-commercial research and educational
purposes only, and the images must not be redistributed (see the upstream READMEs). Nothing in
this repository contains dataset images; the test fixtures are synthetic.